# Is everything in Adelaide 20 minutes away?

The drive from every Greater Adelaide mesh block to eight places, at the speed limit with no traffic (`free`), in
TomTom's 2025 morning and evening rush-hour slowdown (`am`, `pm`), and by public transport arriving 8 to 9 am (`pt`).
Made in [Plotlay](https://www.getplotlay.com). This notebook reads only `../data`.

**Read it carefully.** The rush hour is a city-wide average slowdown, not a traffic forecast; the trips go to the
nearest place of each kind and leave out parking.

In [1]:
import pandas as pd
DATA = '../data'
mb = pd.read_csv(f'{DATA}/mesh_block_minutes.csv', dtype={'mb_code': str, 'sa2_code': str})
PLACES = ['cbd', 'beach', 'airport', 'hospital', 'uni', 'oval', 'hills', 'shops']
total = mb.residents.sum()
print(f'{len(mb):,} mesh blocks, {total:,.0f} residents')

15,797 mesh blocks, 1,386,882 residents


## Within 20 minutes of each place

A mesh block counts when its minutes are 20 or fewer; empty means not reached.

In [2]:
def share(run, place, limit=20):
    m = mb[f'{run}_{place}']
    return (mb.residents * (m <= limit)).sum() / total
table = pd.DataFrame({run: {p: share(run, p) for p in PLACES} for run in ['free', 'am', 'pm', 'pt']})
(table * 100).round(1).sort_values('free', ascending=False)

,free,am,pm,pt
shops,99.4,97.6,97.5,37.4
hospital,93.5,79.1,77.0,4.7
uni,92.3,64.9,64.3,5.0
beach,61.9,40.9,40.2,7.1
oval,51.3,23.9,22.2,1.2
cbd,48.3,21.5,20.0,2.1
airport,39.6,15.1,14.1,0.3
hills,24.6,8.6,8.4,0.2


## All eight

In [3]:
def all_eight(run, limit=20):
    ok = pd.concat([mb[f'{run}_{p}'] <= limit for p in PLACES], axis=1).all(axis=1)
    return (mb.residents * ok).sum() / total
for run in ['free', 'am', 'pm', 'typical', 'pt']:
    print(f'{run:8s} {all_eight(run) * 100:5.1f}% within 20 minutes of all eight')

free      10.1% within 20 minutes of all eight
am         0.0% within 20 minutes of all eight
pm         0.0% within 20 minutes of all eight
typical   28.7% within 20 minutes of all eight
pt         0.0% within 20 minutes of all eight


## How many of the eight, at the speed limit

In [4]:
count = sum((mb[f'free_{p}'] <= 20).astype(int) for p in PLACES)
(mb.groupby(count).residents.sum() / total * 100).round(1).rename('share of residents (%)')

0     0.3
1     0.7
2     3.9
3    23.4
4    17.5
5     9.7
6     8.2
7    26.2
8    10.1
Name: share of residents (%), dtype: float64

## Where it holds: areas where most residents have all eight

In [5]:
sa2 = pd.read_csv(f'{DATA}/sa2.csv', dtype={'sa2_code': str})
sa2[sa2.residents >= 3000].sort_values('share_all_eight_free', ascending=False)[['sa2_name', 'residents', 'share_all_eight_free', 'places_within_20_free', 'mean_min_city_free']].head(10)

,sa2_name,residents,share_all_eight_free,places_within_20_free,mean_min_city_free
0,Goodwood - Millswood,17632,1.000,8.00,10.2
1,Unley - Parkside,20956,1.000,8.00,10.3
2,Colonel Light Gardens,15862,0.999,8.00,13.9
3,Mitcham (SA),16473,0.988,7.99,13.6
4,Adelaide,18223,0.908,7.91,5.0
7,Beaumont - Glen Osmond,10320,0.692,7.57,12.8
5,Toorak Gardens,16635,0.674,7.67,10.1
6,Walkerville,7998,0.604,7.60,10.2
8,Norwood (SA),10854,0.549,7.55,8.2
9,Edwardstown,14552,0.516,7.52,14.5


## Where it fails

In [6]:
sa2[sa2.residents >= 3000].sort_values('places_within_20_free')[['sa2_name', 'residents', 'places_within_20_free', 'mean_min_city_free', 'mean_min_city_am']].head(10)

,sa2_name,residents,places_within_20_free,mean_min_city_free,mean_min_city_am
108,Adelaide Hills,7032,1.01,39.5,62.2
107,Mount Barker Surrounds,6851,1.56,41.6,60.7
106,Lobethal - Woodside,9410,1.76,34.3,51.9
105,Lewiston - Two Wells,6480,1.86,37.0,52.3
104,Nairne,5332,2.00,33.6,47.5
103,Mount Barker,22080,2.12,31.5,44.7
102,Hahndorf - Echunga,4740,2.59,30.2,44.0
100,Gawler - North,9521,2.70,39.5,52.6
99,Gawler - South,21679,2.79,40.1,54.6
98,Willunga,3615,2.82,46.1,67.7
